In [65]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"

# Helper functions to manage messages
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text


In [66]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. 
Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex",
    "solution_criteria": "Key criteria for evaluation the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)

    # Extract JSON from markdown code block if present
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0]
    elif "```" in text:
        text = text.split("```")[1].split("```")[0]

    return json.loads(text.strip())


In [67]:
dataset = generate_dataset()

# write the dataset to a file
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [68]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""

    prompt = f"""
    Please solve the following task:

    {test_case["task"]}
    
    * Respond onlg with Python, JSON, or plain Regex
    * Do not add any comments or commentary, or explaination. Only respond with the code, JSON, or regex that solves the task.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["\n```"])
    return output

In [69]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Criteria you should use to evaluate the solution:
    <solution criteria>
    {test_case["solution_criteria"]}
    </solution criteria>
    
    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
        """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["\n```"])
    return json.loads(eval_text)

In [70]:
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"Unknown format: {format}")

In [71]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(output, test_case)
    
    score = (model_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [72]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score:.2f}")
    
    return results

In [73]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 6.00


In [74]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\n\nlambda_config = {\n    \"FunctionName\": \"S3EventProcessor\",\n    \"Runtime\": \"python3.11\",\n    \"Handler\": \"index.lambda_handler\",\n    \"Role\": \"arn:aws:iam::ACCOUNT_ID:role/lambda-execution-role\",\n    \"Timeout\": 60,\n    \"MemorySize\": 256,\n    \"Environment\": {\n        \"Variables\": {\n            \"DYNAMODB_TABLE_NAME\": \"s3-events-table\",\n            \"CLOUDWATCH_LOG_GROUP\": \"/aws/lambda/s3-event-processor\"\n        }\n    },\n    \"Events\": {\n        \"S3Event\": {\n            \"Type\": \"S3\",\n            \"Properties\": {\n                \"Bucket\": \"my-source-bucket\",\n                \"Events\": [\"s3:ObjectCreated:*\", \"s3:ObjectRemoved:*\"]\n            }\n        }\n    },\n    \"Layers\": [],\n    \"EphemeralStorage\": {\n        \"Size\": 512\n    },\n    \"LoggingConfig\": {\n        \"LogGroup\": \"/aws/lambda/s3-event-processor\",\n        \"LogFormat\": \"JSON\"\n    }\n}\n\nprint(json.dumps(lam